In [1]:
# import libraries
import re
import pandas as pd

In [2]:
# Read the merged CSV file
merged = pd.read_csv(r"C:\Users\telvi\Downloads\AMDARI\montra_guest_exp_sentiment\data\processed\merged_raw.csv")

# Display the first five rows
display(merged.head())


,review_id,booking_id,guest_id,hotel_id,review_text,rating_score,review_date,sentiment_label,review_length,room_type,...,total_amount,stay_duration,customer_type,loyalty_status,country,previous_stays,hotel_name,location,star_rating,total_rooms
0,1,1,6699,18,The overall price was fair for what we paid. M...,3.2,2025-06-02,neutral,80,Suite,...,296.44,1,Leisure,NaN,France,4,The Aberdeen Grand,Aberdeen,4,118
1,2,2,11380,23,The room was nothing out of the ordinary for w...,4.0,2025-01-22,positive,160,Deluxe,...,483.56,4,Business,Gold,Germany,4,Nottingham Park Hotel,Nottingham,4,95
2,3,3,4812,13,"Unfortunately, the pool was outdated — it clea...",1.9,2025-03-13,negative,102,Executive,...,415.59,2,Business,NaN,Ireland,4,The London Boutique,London,4,231
3,4,4,982,12,We were disappointed that the neighborhood was...,2.2,2026-03-18,negative,129,Family Room,...,601.18,3,Leisure,NaN,Nigeria,0,Manchester Metropolitan,Manchester,3,101
4,5,5,615,5,The room service was standard hotel fare for w...,3.3,2025-03-03,neutral,196,Deluxe,...,603.85,5,Group,NaN,United States,9,Glasgow Metropolitan,Glasgow,4,214


In [3]:
# Check the dataset shape
print("Shape:", merged.shape)

Shape: (35000, 23)


# data preprocessing for our target variable "reviews"

In [4]:
# Inspect the quality and characteristics of the review corpus

# Check missing review texts
print("Missing review texts:", merged['review_text'].isna().sum())

# Check duplicate review texts
print("Duplicate reviews:", merged['review_text'].duplicated().sum())

Missing review texts: 0
Duplicate reviews: 464


In [5]:
# Remove duplicate reviews based on review text
review_corpus = (
    merged
    .drop_duplicates(subset=['review_text'])
    .reset_index(drop=True)
)
# Check the result
print("Original number of reviews:", len(merged))
print("Reviews after removing duplicates:", len(review_corpus))
print("Duplicates remaining:", review_corpus['review_text'].duplicated().sum())

Original number of reviews: 35000
Reviews after removing duplicates: 34536
Duplicates remaining: 0


In [6]:
# Remove records with empty or whitespace-only review text
review_corpus = review_corpus[
    review_corpus['review_text'].str.strip().ne('')
].copy()

# Check the result
print("Remaining reviews:", len(review_corpus))
print(
    "Empty reviews remaining:",
    review_corpus['review_text'].str.strip().eq('').sum()
)

Remaining reviews: 34536
Empty reviews remaining: 0


In [7]:
# Calculate the length of each review
review_corpus['text_length'] = review_corpus['review_text'].str.len()

review_corpus['text_length']

0         80
1        160
2        102
3        129
4        196
        ... 
34531    193
34532    144
34533    200
34534    136
34535    150
Name: text_length, Length: 34536, dtype: int64

In [8]:
# display nthe first 5 rows of the review
review_corpus.head()

,review_id,booking_id,guest_id,hotel_id,review_text,rating_score,review_date,sentiment_label,review_length,room_type,...,stay_duration,customer_type,loyalty_status,country,previous_stays,hotel_name,location,star_rating,total_rooms,text_length
0,1,1,6699,18,The overall price was fair for what we paid. M...,3.2,2025-06-02,neutral,80,Suite,...,1,Leisure,NaN,France,4,The Aberdeen Grand,Aberdeen,4,118,80
1,2,2,11380,23,The room was nothing out of the ordinary for w...,4.0,2025-01-22,positive,160,Deluxe,...,4,Business,Gold,Germany,4,Nottingham Park Hotel,Nottingham,4,95,160
2,3,3,4812,13,"Unfortunately, the pool was outdated — it clea...",1.9,2025-03-13,negative,102,Executive,...,2,Business,NaN,Ireland,4,The London Boutique,London,4,231,102
3,4,4,982,12,We were disappointed that the neighborhood was...,2.2,2026-03-18,negative,129,Family Room,...,3,Leisure,NaN,Nigeria,0,Manchester Metropolitan,Manchester,3,101,129
4,5,5,615,5,The room service was standard hotel fare for w...,3.3,2025-03-03,neutral,196,Deluxe,...,5,Group,NaN,United States,9,Glasgow Metropolitan,Glasgow,4,214,196


In [9]:
# Display the first five reviews and their text lengths
display(review_corpus[['review_text', 'text_length']].head())

# Summarise review text length
display(review_corpus['text_length'].describe())

,review_text,text_length
0,The overall price was fair for what we paid. M...,80
1,The room was nothing out of the ordinary for w...,160
2,"Unfortunately, the pool was outdated — it clea...",102
3,We were disappointed that the neighborhood was...,129
4,The room service was standard hotel fare for w...,196


count    34536.000000
mean       172.817437
std         39.220502
min         67.000000
25%        144.000000
50%        172.000000
75%        204.000000
max        280.000000
Name: text_length, dtype: float64

In [10]:
# create a function for the 

def clean_review(text):
    # Convert to string
    text = str(text)

    # Remove HTML tags
    text = re.sub(r'<.*?>', ' ', text)

    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # Remove emojis and non-ASCII characters
    text = text.encode('ascii', 'ignore').decode('ascii')

    # Convert to lowercase
    text = text.lower()

    # Keep letters, numbers, and spaces
    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

## Apply preprocessing

In [14]:
# Apply the clean_review function to review text
review_corpus['cleaned_review_text'] = (review_corpus['review_text'].map(clean_review))

# Display original and cleaned reviews
display(review_corpus[['review_text', 'cleaned_review_text']].head())


,review_text,cleaned_review_text
0,The overall price was fair for what we paid. M...,the overall price was fair for what we paid mi...
1,The room was nothing out of the ordinary for w...,the room was nothing out of the ordinary for w...
2,"Unfortunately, the pool was outdated — it clea...",unfortunately the pool was outdated it clearly...
3,We were disappointed that the neighborhood was...,we were disappointed that the neighborhood was...
4,The room service was standard hotel fare for w...,the room service was standard hotel fare for w...


In [15]:
# Save the cleaned dataset
merged.to_csv("../data/processed/montra_cleaned_data.csv",index=False)

# Print dataset shape
print("Shape:", merged.shape)

Shape: (35000, 24)
